# 06 · A multi-agent supervisor

After this notebook you can build a supervisor that routes work to specialist agents with structured-output routing and `Command` handoffs, run agents as subgraphs, guard against ping-pong loops, count what the team costs, and argue when a team of agents is worth it versus one agent with more tools.

**Time:** ~35 min · **Runs:** offline on CPU in < 45 s (about 24 short LLM calls) · **Needs:** [02 · Conditional edges and loops](02_conditional_edges_and_loops.ipynb), [05 · ReAct tool-calling agent](05_react_tool_calling_agent.ipynb)

## Why this matters in interviews

- "How would you set up a multi-agent system, and when should you?" is a staple. The strong answer is sceptical: start with one agent and a router, split only for a named reason (different tools, permissions or contexts), and show you know the costs.
- Supervisor design questions probe the details: how the supervisor decides, what each worker sees, what the handoff looks like, and what stops two agents handing work back and forth forever.
- "Five specialised agents scored worse than one general agent" questions want the mechanisms: lossy handoffs, error compounding, extra calls. This notebook measures the call count.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ag12` | How do you set up a multi-agent system, and when should you? |
| `ag27` | What is a supervisor agent and how does it decide what to delegate? |
| `ag13` | What goes wrong in multi-agent systems? |
| `ag46` | You built five specialised agents. Together they score worse than one general agent on the same base model. Why? |
| `ag31` | What is a router agent, and why is it the highest-leverage pattern? |
| `tl17` | Compare CrewAI, AutoGen and LangGraph. |
| `tl22` | Give me the four guards that stop an agent costing $400 in one run. |

## 1. Setup

How the offline stand-in behaves here, so the outputs make sense: for **structured output** it fills an enum with the first allowed value that appears in the user message, else the first allowed value; for **tools** it picks the best keyword match. So offline, the supervisor effectively takes the first allowed option. With a real model the same code routes by meaning.

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

In [ ]:
import operator
import re
from collections import Counter
from typing import Annotated, Literal, TypedDict

import matplotlib.pyplot as plt
from langchain.agents import create_agent
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.types import Command
from pydantic import Field, create_model
from llm_setup import langchain_chat_model

llm = langchain_chat_model(temperature=0)
print("chat model:", MODEL, "| provider:", PROVIDER)

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def draw_graph(app, title=None, xray=False):
    """Draw a compiled LangGraph left to right: solid = fixed edge, dashed = conditional edge,
    arcs above = skip ahead, arcs below = loop back. Red text = the path-map label."""
    g = app.get_graph(xray=xray)
    edges = [(e.source, e.target, e.conditional, e.data) for e in g.edges]
    col, frontier = {"__start__": 0}, ["__start__"]
    while frontier:                                     # column = hops from START (BFS)
        nxt = []
        for n in frontier:
            for s, t, _, _ in edges:
                if s == n and t not in col and t != "__end__":
                    col[t] = col[n] + 1; nxt.append(t)
        frontier = nxt
    for n in g.nodes:
        col.setdefault(n, max(col.values()) + 1)
    col["__end__"] = max(c for n, c in col.items() if n != "__end__") + 1
    rows = {}
    for n in g.nodes:
        rows.setdefault(col[n], []).append(n)
    pos = {n: (c * 2.4, (len(ns) - 1) / 2 - i) for c, ns in rows.items() for i, n in enumerate(ns)}
    nrow = max(len(v) for v in rows.values())
    fig, ax = plt.subplots(figsize=(min(11, 2.0 * len(rows) + 0.6), min(4.5, 0.9 * nrow + 1.4)))
    boxes = {}
    for n, (x, y) in pos.items():
        parts = [{"__start__": "START", "__end__": "END"}.get(p, p) for p in n.split(":")]
        fc = {"START": "#d3f9d8", "END": "#ffe3e3"}.get(parts[-1], "#e7f0ff")
        boxes[n] = FancyBboxPatch((x - 0.8, y - 0.28), 1.6, 0.56, boxstyle="round,pad=0.05", fc=fc, ec="#3b5bdb")
        ax.add_patch(boxes[n])
        ax.text(x, y, "\n".join(parts), ha="center", va="center", fontsize=8.5 if len(parts) == 1 else 7.5)
    pairs = {(s, t) for s, t, _, _ in edges}
    for s, t, cond, data in edges:
        (x1, y1), (x2, y2) = pos[s], pos[t]
        span = col[t] - col[s]
        if (t, s) in pairs and abs(span) == 1:
            rad = 0.2                                   # two-way pair: split the arrows apart
        elif span == 1 or (span == 0 and s != t):
            rad = 0.0
        else:
            rad = -0.3 if span > 1 else -0.45           # skip ahead above, loop back below
        ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), patchA=boxes[s], patchB=boxes[t], arrowstyle="-|>",
                                     mutation_scale=12, lw=1.3, color="#495057", linestyle="--" if cond else "-",
                                     connectionstyle=f"arc3,rad={rad}"))
        if data is not None and str(data) != t:
            mx, my = (x1 + x2) / 2 + 0.5 * rad * (y2 - y1), (y1 + y2) / 2 - 0.5 * rad * (x2 - x1)
            ax.text(mx, my + 0.1, str(data), ha="center", fontsize=7.5, color="#c92a2a")
    xs = [p[0] for p in pos.values()]; ys = [p[1] for p in pos.values()]
    ax.set_xlim(min(xs) - 1.0, max(xs) + 1.0); ax.set_ylim(min(ys) - 0.75, max(ys) + 0.85)
    ax.axis("off"); ax.set_title(title or "graph", fontsize=10)
    plt.tight_layout(); plt.show()

## 2. The team and its tools

**In plain English:** a supervisor is a project manager. It holds the customer's request, decides which specialist should act next, hands them a brief, collects the result, and decides again until the job is done. The specialists are agents with **different tools**, which is the honest reason to split work at all:

| Worker | Tools | Job |
|---|---|---|
| `researcher` | `search_kb` over a tiny knowledge base | find the facts: prices, policies |
| `calculator` | `quote_total` (deterministic arithmetic) | price the quote; never let a model do sums in its head |
| `writer` | none | turn the findings into a customer reply |

In [ ]:
KB = {
    "pricing": "The Pro plan costs $30 per seat per month. The Basic plan costs $10 per seat per month.",
    "refunds": "Refunds: you can cancel at any time. A full refund is available within 14 days of purchase; "
               "after that, cancelling early stops renewal but there is no cash refund.",
    "seats": "Seats can be added or removed at any time; changes apply from the next billing cycle.",
    "support": "Support is available 9am to 6pm CET on weekdays; Pro customers also get chat support.",
    "security": "All data is encrypted at rest and in transit; SSO is available on the Pro plan.",
}
STOP = set("a an the is are to of on for and or in at it can be if they that this how much will what "
           "get want wants customer customers find need needed request".split())
words = lambda text: {w.rstrip("s") for w in re.findall(r"[a-z]+", text.lower()) if w not in STOP}

@tool
def search_kb(query: str) -> str:
    """Search the company knowledge base for plan prices, seat rules and refund or cancellation policies."""
    ranked = sorted(KB, key=lambda k: -len(words(query) & words(KB[k])))
    return " ".join(f"[{k}] {KB[k]}" for k in ranked[:2])

PRICE = {"pro": 30, "basic": 10}

@tool
def quote_total(plan: Literal["pro", "basic"], seats: int, months: int) -> str:
    """Compute the total price for a quote: plan, number of seats and number of months."""
    return f"{seats} seats x {months} months x ${PRICE[plan]} = ${seats * months * PRICE[plan]:,} on the {plan} plan"

REQUEST = ("A customer wants 12 seats on the Pro plan for 3 months. How much will that cost, "
           "and can they get a refund if they cancel early?")
print(search_kb.invoke({"query": REQUEST}))
print(quote_total.invoke({"plan": "pro", "seats": 12, "months": 3}))
assert search_kb.invoke({"query": REQUEST}).startswith("[pricing]") and "refunds" in search_kb.invoke({"query": REQUEST})

Each specialist is a full agent (a compiled LangGraph graph from `create_agent`, notebook 05) with its own prompt and tools. The writer needs no tools, so it is a single model call.

In [ ]:
researcher_agent = create_agent(llm, [search_kb], name="researcher",
                                system_prompt="You are a researcher. Use search_kb and report only facts you found.")
calculator_agent = create_agent(llm, [quote_total], name="calculator",
                                system_prompt="You price quotes. Always use quote_total; never do arithmetic yourself.")
print(type(researcher_agent).__name__, "| nodes:", [n for n in researcher_agent.get_graph().nodes if not n.startswith("__")])

## 3. Shared state: a blackboard plus a route log

The team shares one state. `notes` is a **blackboard**: each worker writes its result under its own name, and everyone reads it (a dict-merging reducer from notebook 01). The supervisor never sees the workers' internal tool calls, only their notes: that keeps its context small.

In [ ]:
def merge_dicts(current: dict, update: dict) -> dict:
    return {**(current or {}), **update}

class TeamState(TypedDict):
    request: str
    notes: Annotated[dict, merge_dicts]          # the blackboard: worker name -> result
    route_log: Annotated[list, operator.add]     # every supervisor decision, for debugging
    steps: int                                   # supervisor turns so far
    final: str

WORKERS = ["researcher", "calculator", "writer"]
MAX_STEPS = 5

## 4. The supervisor: structured-output routing + `Command`

The supervisor asks the model for a **typed decision**, `next: Literal[...]`, so the answer is always a valid destination, never free text to parse. It then returns `Command(goto=next)`. Three guards live in plain code around the model call:

1. **Done-check in code:** if the writer has produced the reply, finish without asking the model.
2. **Step budget:** at `MAX_STEPS` supervisor turns, stop with a named degraded outcome.
3. **Scoped options:** each worker may run at most once per request, so the enum only offers workers that have not run yet (plus `FINISH`). That bounds the cost of a run and makes ping-pong impossible by construction. Offline, the stand-in takes the first allowed option, which happens to be the natural order; a real model chooses by content and may finish early.

In [ ]:
SUPERVISOR_PROMPT = SystemMessage(
    "You supervise a support team. Workers: researcher (searches the knowledge base for prices and policies), "
    "calculator (prices quotes with a tool), writer (writes the customer reply from the findings). "
    "Choose who acts next. Choose FINISH only when nothing useful is left to do.")

def route_model(options):
    """A Pydantic model whose only field is an enum of the options that are legal right now."""
    return create_model("Route", next=(Literal[tuple(options)], Field(description="who acts next")))

def supervisor(state: TeamState) -> Command[Literal["researcher", "calculator", "writer", "__end__"]]:
    if state.get("final"):
        return Command(goto=END, update={"route_log": ["FINISH (done-check in code)"]})
    if state["steps"] >= MAX_STEPS:
        return Command(goto=END, update={"final": "DEGRADED: step budget exhausted. Findings: " + str(state["notes"]),
                                         "route_log": ["STOP (step budget)"]})
    options = [w for w in WORKERS if w not in state["notes"]] + ["FINISH"]
    work = "\n".join(f"- {name}: {result}" for name, result in state["notes"].items()) or "(nothing yet)"
    decision = llm.with_structured_output(route_model(options)).invoke(
        [SUPERVISOR_PROMPT, HumanMessage(f"Request: {state['request']}\n\nWork so far:\n{work}")])
    goto = END if decision.next == "FINISH" else decision.next
    return Command(goto=goto, update={"steps": state["steps"] + 1, "route_log": [decision.next]})

## 5. Workers are agents, wrapped as nodes

Each worker node **passes a scoped brief** into its agent (message passing: the agent sees one message, not the team's whole history), takes the agent's final answer, writes it to the blackboard, and hands control back with `Command(goto="supervisor")`.

In [ ]:
def researcher(state: TeamState) -> Command[Literal["supervisor"]]:
    brief = f"Find the facts needed for this request: {state['request']}"
    result = researcher_agent.invoke({"messages": [HumanMessage(brief)]})["messages"][-1].content
    return Command(goto="supervisor", update={"notes": {"researcher": result}})

def calculator(state: TeamState) -> Command[Literal["supervisor"]]:
    brief = f"Compute the total price for this request: {state['request']}"
    result = calculator_agent.invoke({"messages": [HumanMessage(brief)]})["messages"][-1].content
    return Command(goto="supervisor", update={"notes": {"calculator": result}})

def writer(state: TeamState) -> Command[Literal["supervisor"]]:
    findings = " ".join(state["notes"].get(k, "") for k in ("calculator", "researcher"))
    reply = llm.invoke(f"Summarise these findings as a short, friendly reply to the customer.\n\n{findings}").content
    return Command(goto="supervisor", update={"notes": {"writer": "reply drafted"}, "final": reply})

tb = StateGraph(TeamState)
tb.add_node("supervisor", supervisor)
for name, fn in [("researcher", researcher), ("calculator", calculator), ("writer", writer)]:
    tb.add_node(name, fn)
tb.add_edge(START, "supervisor")
team = tb.compile()
draw_graph(team, "Supervisor team: every worker reports back to the supervisor (Command handoffs)")

## 6. Run it, and count what it costs

A callback handler sees every model call. LangGraph tags each call with metadata including `langgraph_checkpoint_ns`, whose first segment is the **top-level node** the call came from, even when the call happened deep inside a worker's own subgraph. That gives per-agent cost accounting for free.

In [ ]:
class CostMeter(BaseCallbackHandler):
    """Counts LLM calls and tokens per top-level node (= per agent)."""
    def __init__(self):
        self.calls, self.tokens, self._who = Counter(), Counter(), {}

    def on_chat_model_start(self, serialized, messages, *, run_id, metadata=None, **kwargs):
        ns = (metadata or {}).get("langgraph_checkpoint_ns") or (metadata or {}).get("langgraph_node", "?")
        self._who[run_id] = ns.split(":")[0]
        self.calls[self._who[run_id]] += 1

    def on_llm_end(self, response, *, run_id, **kwargs):
        usage = (response.llm_output or {}).get("token_usage") or {}
        self.tokens[self._who.get(run_id, "?")] += usage.get("total_tokens", 0)

team_meter = CostMeter()
result = team.invoke({"request": REQUEST, "notes": {}, "route_log": [], "steps": 0, "final": ""},
                     {"callbacks": [team_meter]})
print("route      :", " -> ".join(result["route_log"]))
for name in ("researcher", "calculator"):
    print(f"{name:11s}:", result["notes"][name][:110])
print("final reply:", result["final"][:160])
print("LLM calls  :", dict(team_meter.calls), "| total", sum(team_meter.calls.values()))
assert result["final"] and not result["final"].startswith("DEGRADED")
assert result["route_log"][-1].startswith("FINISH") and len(result["route_log"]) <= MAX_STEPS + 1
assert "1,080" in result["notes"]["calculator"]

The quote came from the tool (`$1,080`), not from a model doing arithmetic. The route log is the first thing you read when a team misbehaves: it is the supervisor's trajectory. Offline, the writer's reply is the stand-in's extractive "Summary:"; a real model writes a proper customer message.

## 7. The ping-pong failure and the step guard

Remove the scoped options and the done-check, and nothing stops the supervisor choosing the same worker again and again. Offline the stand-in does exactly that (the worker's name now appears in "Work so far", so it is picked again), a caricature of the real failure where two agents keep handing the task back and forth. Only the step budget ends it:

In [ ]:
def naive_supervisor(state: TeamState) -> Command[Literal["researcher", "calculator", "writer", "__end__"]]:
    if state["steps"] >= 3:                                         # the only guard left
        return Command(goto=END, update={"final": "DEGRADED: step budget exhausted", "route_log": ["STOP (step budget)"]})
    work = "\n".join(f"- {name}: {result}" for name, result in state["notes"].items()) or "(nothing yet)"
    decision = llm.with_structured_output(route_model(WORKERS + ["FINISH"])).invoke(
        [SUPERVISOR_PROMPT, HumanMessage(f"Request: {state['request']}\n\nWork so far:\n{work}")])
    goto = END if decision.next == "FINISH" else decision.next
    return Command(goto=goto, update={"steps": state["steps"] + 1, "route_log": [decision.next]})

nb = StateGraph(TeamState)
nb.add_node("supervisor", naive_supervisor)
for name, fn in [("researcher", researcher), ("calculator", calculator), ("writer", writer)]:
    nb.add_node(name, fn)
nb.add_edge(START, "supervisor")
naive_meter = CostMeter()
naive = nb.compile().invoke({"request": REQUEST, "notes": {}, "route_log": [], "steps": 0, "final": ""},
                            {"callbacks": [naive_meter]})
print("route    :", " -> ".join(naive["route_log"]))
print("outcome  :", naive["final"])
print("LLM calls:", dict(naive_meter.calls), "| total", sum(naive_meter.calls.values()))
assert naive["final"].startswith("DEGRADED")

With a real model this loop is rarer, which makes it worse: it passes the demo and then happens in production on an ambiguous request. The defences are the ones from notebook 02, applied per team: a step budget with a named outcome, repeat detection (the same worker twice in a row with no new information), a done-check in code, and `recursion_limit` as the backstop.

## 8. Is the team worth it? One agent with all the tools

The same request, handled by a single agent that has both tools:

In [ ]:
solo_agent = create_agent(llm, [search_kb, quote_total],
                          system_prompt="You are a support agent. Use the tools, then answer the customer.")
solo_meter = CostMeter()
solo = solo_agent.invoke({"messages": [HumanMessage(REQUEST)]}, {"callbacks": [solo_meter]})
tools_used = [m.name for m in solo["messages"] if m.type == "tool"]
print("tools used :", tools_used)
print("answer     :", solo["messages"][-1].content[:160])
print("LLM calls  :", dict(solo_meter.calls), "| total", sum(solo_meter.calls.values()))
assert set(tools_used) == {"search_kb", "quote_total"}
assert sum(solo_meter.calls.values()) < sum(team_meter.calls.values())

In [ ]:
runs = {"one agent, both tools": solo_meter, "supervisor team": team_meter, "team, ping-pong (guarded)": naive_meter}
agents = ["supervisor", "researcher", "calculator", "writer", "model"]
colors = {"supervisor": "#495057", "researcher": "#3b5bdb", "calculator": "#2f9e44", "writer": "#e8590c", "model": "#868e96"}
fig, ax = plt.subplots(figsize=(8.5, 3.2))
for i, (label, meter) in enumerate(runs.items()):
    left = 0
    for a in agents:
        if meter.calls.get(a):
            ax.barh(i, meter.calls[a], left=left, color=colors[a], label=a if i == 1 or a == "model" else None)
            left += meter.calls[a]
ax.set_yticks(range(len(runs)), list(runs)); ax.set_xlabel("LLM calls for one request")
ax.set_title("Same request, same model: the team costs several times more calls")
handles, labels = ax.get_legend_handles_labels()
uniq = dict(zip(labels, handles)); ax.legend(uniq.values(), uniq.keys(), fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()

("model" is the node name inside `create_agent`, i.e. the single agent's own calls.) For this task the team adds calls, latency and two handoffs where information can be lost, and buys nothing the single agent lacked: the task needs two tools and no separation of permissions or context. That is the usual outcome, and why the default answer is **one agent, more tools, plus a router in front**.

| Choose one agent with more tools when... | Choose a multi-agent design when... |
|---|---|
| the task needs a handful of tools (under ~10–15) that fit one prompt | tool sets are large or conflicting, so one prompt would confuse the model |
| all steps share the same context and mutable state (e.g. coding) | subtasks are independent and read-only (search many sources in parallel) |
| latency and cost per request matter | workers need **different permissions** (a refund agent vs a public FAQ agent) |
| you want one trajectory to evaluate and debug | contexts would collide or leak (customer data vs internal code) |
| errors must not compound across hops (0.9 × 0.9 × 0.9 = 0.73) | you need to evaluate and improve each specialist separately |

## 9. Two other handoff styles

**Agents as tools.** Instead of routing, the supervisor is itself a ReAct agent whose "tools" call the specialists. It stays in control, results come back as tool messages, and parallel calls come for free. This is the pattern LangChain's current docs recommend for most supervisor setups.

In [ ]:
@tool
def ask_researcher(question: str) -> str:
    """Ask the research agent to look up facts in the knowledge base: plan prices, seats, refund and cancellation policies."""
    return researcher_agent.invoke({"messages": [HumanMessage(question)]})["messages"][-1].content

@tool
def ask_calculator(request: str) -> str:
    """Ask the pricing agent to compute the total cost of a quote for a plan, number of seats and months."""
    return calculator_agent.invoke({"messages": [HumanMessage(request)]})["messages"][-1].content

boss = create_agent(llm, [ask_researcher, ask_calculator], system_prompt="Delegate to your specialists, then answer.")
tools_meter = CostMeter()
out = boss.invoke({"messages": [HumanMessage(REQUEST)]}, {"callbacks": [tools_meter]})
print("delegated to:", [m.name for m in out["messages"] if m.type == "tool"])
print("LLM calls   :", sum(tools_meter.calls.values()))
assert {m.name for m in out["messages"] if m.type == "tool"} == {"ask_researcher", "ask_calculator"}

**Peer handoffs with `Command(graph=Command.PARENT)`.** A node *inside* a worker's subgraph can jump to a sibling in the parent graph, which is how "transfer to the billing agent" tools are implemented in `langgraph-supervisor` and `langgraph-swarm`:

In [ ]:
class Log(TypedDict):
    log: Annotated[list, operator.add]

def transfer_to_billing(state: Log) -> Command:
    return Command(goto="billing", graph=Command.PARENT, update={"log": ["sales: this is a billing question, handing off"]})

sales_team = StateGraph(Log)
sales_team.add_node("triage", transfer_to_billing)
sales_team.add_edge(START, "triage")

parent = StateGraph(Log)
parent.add_node("sales", sales_team.compile())
parent.add_node("billing", lambda s: {"log": ["billing: handled the refund question"]})
parent.add_edge(START, "sales"); parent.add_edge("billing", END)
print(parent.compile().invoke({"log": []})["log"])

The prebuilt packages (**not installed here**; the code below is their documented shape, not run):

```python
# pip install langgraph-supervisor
from langgraph_supervisor import create_supervisor
research = create_react_agent(model, [search_kb], name="research_expert", prompt="You look things up.")
pricing = create_react_agent(model, [quote_total], name="pricing_expert", prompt="You price quotes.")
app = create_supervisor([research, pricing], model=model,
                        prompt="You manage a research expert and a pricing expert.").compile()

# pip install langgraph-swarm        (no supervisor: agents hand off to each other directly)
from langgraph_swarm import create_swarm, create_handoff_tool
alice = create_react_agent(model, [search_kb, create_handoff_tool(agent_name="Bob")], name="Alice", prompt="...")
bob = create_react_agent(model, [quote_total, create_handoff_tool(agent_name="Alice")], name="Bob", prompt="...")
app = create_swarm([alice, bob], default_active_agent="Alice").compile(checkpointer=InMemorySaver())
```

A swarm remembers which agent was last active per thread, so the user keeps talking to the specialist. It is also the easiest topology to loop, so it needs the same step budget.

## 10. Shared state vs message passing, measured

What should the supervisor see? If a worker's subgraph is added **directly as a node** on a shared `messages` channel, every internal tool call and tool result flows into the team's history. The wrapper nodes above pass only a brief in and a result out.

In [ ]:
shared = StateGraph(MessagesState)
shared.add_node("researcher", researcher_agent)          # the compiled agent itself is the node
shared.add_edge(START, "researcher"); shared.add_edge("researcher", END)
shared_msgs = shared.compile().invoke({"messages": [HumanMessage(REQUEST)]})["messages"]

brief_in_result_out = researcher(state={"request": REQUEST, "notes": {}, "route_log": [], "steps": 0, "final": ""})
print("shared messages channel :", [m.type for m in shared_msgs], f"({sum(len(str(m.content)) for m in shared_msgs)} chars)")
print("message passing (notes) :", f"1 note ({len(brief_in_result_out.update['notes']['researcher'])} chars)")
assert len(shared_msgs) > 1
draw_graph(shared.compile(), "A subgraph added as a node (xray=True shows its inside)", xray=True)

| | Shared state (one message list, or a blackboard) | Message passing (brief in, result out) |
|---|---|---|
| What a worker sees | everything so far | only its brief |
| Supervisor context | grows with every worker's tool calls | stays small: one result per worker |
| Information loss at handoffs | low: nothing is summarised | real risk: the brief or the result may drop what mattered |
| Contamination | workers can be confused by each other's traffic | contexts are isolated |
| Debugging | one long transcript | per-worker traces plus the route log |
| Good default | a **typed blackboard** (`notes`) for artefacts, plus message passing for conversation | |

Pass **artefacts, not summaries**: a typed quote, the retrieved passages with their ids, a file path, so the next agent works from the evidence.

## 11. Hierarchical teams

When one supervisor would need too many workers, give it **teams**: each team is a compiled graph with its own supervisor, added as a node of the top-level graph. Pure Python here, so the structure is easy to see:

In [ ]:
class TeamLocalState(TypedDict):                 # what a team sees: no top-level log
    request: str
    notes: Annotated[dict, merge_dicts]

class OrgState(TeamLocalState):
    log: Annotated[list, operator.add]           # only the top level writes this

def sub_supervisor(order):
    """A team lead that sends work to the first team member who has not reported yet."""
    def lead(state: TeamLocalState):
        todo = [m for m in order if m not in state["notes"]]
        return Command(goto=todo[0] if todo else END)
    return lead

def member(name):
    return lambda s: Command(goto="lead", update={"notes": {name: f"done ({len(s['notes']) + 1})"}})

def team_graph(members):
    g = StateGraph(TeamLocalState)
    g.add_node("lead", sub_supervisor(members), destinations=tuple(members) + (END,))
    for m in members:
        g.add_node(m, member(m), destinations=("lead",))
    g.add_edge(START, "lead")
    return g.compile()

def top_supervisor(state: OrgState) -> Command[Literal["research_team", "writing_team", "__end__"]]:
    if "kb_search" not in state["notes"]:
        return Command(goto="research_team", update={"log": ["top -> research_team"]})
    if "editor" not in state["notes"]:
        return Command(goto="writing_team", update={"log": ["top -> writing_team"]})
    return Command(goto=END, update={"log": ["top -> FINISH"]})

org = StateGraph(OrgState)
org.add_node("top", top_supervisor)
org.add_node("research_team", team_graph(["kb_search", "web_search"]))
org.add_node("writing_team", team_graph(["drafter", "editor"]))
org.add_edge(START, "top"); org.add_edge("research_team", "top"); org.add_edge("writing_team", "top")
org_app = org.compile()
org_out = org_app.invoke({"request": REQUEST, "notes": {}, "log": []})
print("top-level log:", " | ".join(org_out["log"]))
print("notes        :", org_out["notes"])
assert org_out["log"] == ["top -> research_team", "top -> writing_team", "top -> FINISH"]
draw_graph(org_app, "Hierarchical teams: the top supervisor delegates to two team graphs")

A subtle bug avoided here: the team graphs use their own schema **without** the `log` key. A subgraph added as a node returns its whole state for the keys it shares with the parent, so if a team shared an append-only `log` it would hand back the entire list, parent entries included, and the parent's `operator.add` would append it all a second time. Share only keys whose reducer is safe to re-apply (like the dict merge on `notes`), or give subgraphs their own keys.

Each level only sees its own direct reports, which keeps every supervisor's choice (and context) small. The cost: every level adds routing calls and another place where information is summarised, so use a hierarchy only when a flat team has genuinely too many members to choose between.

## Try it yourself

**1.** Set `MAX_STEPS = 1` and rerun the team. Show that the run ends with the degraded outcome, and that the findings gathered before the stop are still reported.

In [ ]:
# Solution — try it yourself first, then run this
MAX_STEPS = 1
short = team.invoke({"request": REQUEST, "notes": {}, "route_log": [], "steps": 0, "final": ""})
print(short["route_log"]); print(short["final"][:140])
assert short["final"].startswith("DEGRADED") and "researcher" in short["final"]
MAX_STEPS = 5

**2.** Turn the call counts into money. Using the token counts in `team_meter` and `solo_meter`, estimate the cost per request at an assumed $0.40 per million input tokens and $1.60 per million output tokens (check your provider's current price list). For simplicity, price all tokens at the output rate as an upper bound, and report the ratio team / solo.

In [ ]:
# Solution — try it yourself first, then run this
UPPER_PRICE_PER_TOKEN = 1.60 / 1_000_000
for label, meter in [("solo", solo_meter), ("team", team_meter)]:
    toks = sum(meter.tokens.values())
    print(f"{label}: {toks} tokens, at most ${toks * UPPER_PRICE_PER_TOKEN:.6f} per request")
ratio = sum(team_meter.tokens.values()) / max(1, sum(solo_meter.tokens.values()))
print(f"team / solo token ratio: {ratio:.1f}x")
assert ratio > 1

**3.** Artefacts, not summaries: change the calculator node so it stores the **raw `quote_total` tool output** (from the calculator agent's `ToolMessage`) in the blackboard instead of the agent's final prose, and check the stored note starts with "12 seats".

In [ ]:
# Solution — try it yourself first, then run this
def calculator_artifact(state: TeamState) -> Command[Literal["supervisor"]]:
    msgs = calculator_agent.invoke({"messages": [HumanMessage(f"Compute the total price for: {state['request']}")]})["messages"]
    raw = next(m.content for m in msgs if m.type == "tool" and m.name == "quote_total")
    return Command(goto="supervisor", update={"notes": {"calculator": raw}})

upd = calculator_artifact({"request": REQUEST, "notes": {}, "route_log": [], "steps": 0, "final": ""}).update
print(upd["notes"]["calculator"])
assert upd["notes"]["calculator"].startswith("12 seats")

## Say it in an interview

**30-second answer (when multi-agent):** "I start with one agent and a router in front. I split into agents only for a reason I can name: tool sets too large for one prompt, different permissions, contexts that must not mix, or independent read-only work I can parallelise. Every extra agent is another context window, another handoff that can lose information and more model calls: in the notebook the same request took several times more calls as a team than as one agent with both tools."

**30-second answer (supervisor design):** "The supervisor owns the goal and returns a typed routing decision, a structured-output enum of the workers allowed right now, and hands off with `Command(goto=...)`. Workers get a scoped brief and write artefacts to a typed blackboard; the supervisor sees results, not their internal traffic. Guards live in code: a done-check, a step budget with a degraded outcome, repeat detection, per-agent cost accounting from callback metadata, and `recursion_limit` as the backstop."

**Facts to have ready:**
- Topologies: supervisor/worker, agents-as-tools, peer handoff (swarm), hierarchical teams, blackboard.
- In LangGraph a worker can be a compiled graph added as a node (shares state keys) or called inside a wrapper node (own schema, message passing); `Command(graph=Command.PARENT)` jumps from a subgraph to a parent node.
- Error compounding: 90% per hop over five hops is about 59% end to end.
- `langgraph-supervisor` (`create_supervisor`) and `langgraph-swarm` (`create_swarm`, `create_handoff_tool`) package these patterns.

**Follow-ups they ask:** *"Five specialists did worse than one generalist. Why?"* Lossy summaries at every handoff, compounding errors, unrecoverable misroutes, contradictory beliefs without shared state, and nobody owning the final answer; fix by passing artefacts, giving one agent ownership, and instrumenting per hop. *"CrewAI or AutoGen?"* Conversation-first frameworks get to a demo fastest; a graph gives explicit, checkpointed, testable control flow.

**Traps:** justifying agents with an org chart instead of tools/permissions, letting the supervisor's context accumulate every worker's transcript, free-text routing that must be parsed, and no per-agent cost visibility.

## Next

- [Multi-agent patterns](../15_agentic_ai/05_multi_agent_patterns.ipynb) · [ReAct and plan-and-execute](../15_agentic_ai/03_react_and_plan_and_execute.ipynb) · [agent evaluation](../12_evaluation/04_agent_evaluation.ipynb) · [latency percentiles and cost](../16_production/05_latency_percentiles_p50_p95_and_cost.ipynb)
- Back to [01 · State, nodes and edges](01_state_nodes_and_edges.ipynb) for the building blocks.
- Theory: [LangGraph course](../../langgraph/index.html) (chapter 12, subgraphs and handoffs), [agentic AI flow](../../agentic_ai_flow/index.html), [AI system design concepts](../../ai_system_design_concepts/index.html), [interview bank](../../ai_interview_prep/index.html).